# MySQL → OneBill Migration (Fixed + Profiled)
Bug fixes and profiling instrumentation applied. See inline comments for each change.

In [66]:
# %pip install mysql-connector-python
# %pip install sqlalchemy
# %pip install python-dotenv
import json
import pandas as pd
from sqlalchemy import create_engine
import requests
import logging
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import threading
import os
from collections import defaultdict

from dotenv import load_dotenv
load_dotenv(override=True)  # override=True ensures .env values take precedence


2026-05-05 14:17:13,475 [WARNING] python-dotenv could not parse statement starting at line 1
2026-05-05 14:17:13,477 [WARNING] python-dotenv could not parse statement starting at line 5


True

## Configuration

In [67]:
MAX_WORKERS = 20
TOKEN_TTL_SECONDS = 3500  # Refresh token 100s before expiry (typical OAuth TTL is 3600s)

db_url = f'mysql+mysqlconnector://{os.environ["DB_USERNAME"]}:{os.environ["DB_PASSWORD"]}@{os.environ["DB_HOST"]}/bi_custom_views'
baseUrl = 'https://sandbox-sg.onebillsoftware.com'
accessTokenURL = f'{baseUrl}/oauth/token'


## Get MySQL Data

In [78]:
query = """
    SELECT
        `AccountCode`,
        `AccountName`,
        `AccountType`,
        CASE
            WHEN `AccountType` = 'Actrix Residential' THEN 302
            WHEN `AccountType` = 'Residential'        THEN 303
            WHEN `AccountType` = 'HD Consumer'        THEN 305
            WHEN `AccountType` = 'Staff'              THEN 203
        END AS `OneBill_AccountType`,
        `AccountManager`,
        `CreatedDate`,
        CASE
            WHEN `EmailAddresses` IS NULL OR `EmailAddresses` = '' THEN 'someone@gmail.com'
            ELSE `EmailAddresses`
        END AS `EmailAddresses`,
        `firstName`,
        `lastName`,
        `phoneHome`,
        `mobile`,
        CASE
            WHEN `addr1` IS NULL OR `addr1` = '' THEN '1 Somewhere Place'
            ELSE `addr1`
        END AS `Address1`,
        `addr2` AS `Address2`,
        CASE
            WHEN `suburb` IS NULL OR `suburb` = '' THEN 'N/A'
            ELSE `suburb`
        END AS `Suburb`,
        CASE
            WHEN `city` IS NULL OR `city` = '' THEN 'Auckland'
            ELSE `city`
        END AS `City`,
        CASE
            WHEN `postcode` IS NULL OR `postcode` = '' THEN 0000
            ELSE `postcode`
        END AS `Postcode`,
        `dob` AS `DateOfBirth`
    FROM bi_custom_views.reporting_account_allColumns
    WHERE `AccountSegment` = 'Consumer'
    AND `dob` IS NOT NULL
    ORDER BY `AccountCode` DESC;
"""

engine = create_engine(db_url)
df = pd.read_sql(query, con=engine)
df = df.head()   # <-- REMOVE this line in production; it limits to 30,000 rows only
print(f'Loaded {len(df):,} rows from MySQL')


Loaded 5 rows from MySQL


## Log Setup

In [79]:
log_filename = f'migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


## Token Manager (Thread-Safe)

**Bug #1 (Critical — was the main bottleneck):** `create_onebill_account()` was calling
`get_AccessToken()` on every single request, inside the thread. With 20 workers each
making hundreds of calls, this meant potentially thousands of redundant token
round-trips to the OAuth server — serialised through the GIL and hammering the
auth endpoint. A token is valid for ~1 hour; we only need to fetch it once (and
refresh it proactively before it expires).

In [80]:
class TokenManager:
    """Thread-safe bearer token cache with proactive refresh."""

    def __init__(self):
        self._lock = threading.Lock()
        self._token: str | None = None
        self._expires_at: datetime = datetime.min

    def get_token(self) -> str:
        with self._lock:           # only one thread refreshes at a time
            if datetime.now() >= self._expires_at:
                self._refresh()    # others wait at the lock, then see valid token
            return self._token

    def _refresh(self):
        logger.info('Refreshing OAuth token...')
        token_data = {
            'grant_type':    'password',
            'client_id':     os.environ['CLIENT_ID'],
            'client_secret': os.environ['CLIENT_SECRET'],
            'username':      os.environ['API_USERNAME'],
            'password':      os.environ['API_PASSWORD'],
        }
        response = requests.post(
            accessTokenURL,
            data=token_data,
            headers={'Content-Type': 'application/x-www-form-urlencoded'}
        )
        response.raise_for_status()
        payload = response.json()
        self._token = payload['access_token']
        ttl = payload.get('expires_in', TOKEN_TTL_SECONDS)
        self._expires_at = datetime.now() + timedelta(seconds=ttl - 100)  # 100s safety margin
        logger.info('Token refreshed; valid until %s', self._expires_at.strftime('%H:%M:%S'))

token_manager = TokenManager()


## Payload Builder

In [86]:
def build_account_payload(row: pd.Series) -> str:
    """Map a DataFrame row to the OneBill Account API payload."""
    row = row.where(pd.notna(row), None).to_dict()

    def serialize_date(value, fmt=None):
        if value is None:
            return None
        if hasattr(value, 'isoformat'):
            return value.strftime(fmt) if fmt else value.isoformat()
        if fmt:
            try:
                return datetime.strptime(str(value), '%Y-%m-%d').strftime(fmt)
            except ValueError:
                return str(value)
        return str(value)

    return json.dumps({
        'accountNumber':       row['AccountCode'],
        'accountName':         row['AccountName'],
        'accountType':         '1001',
        'accountSubType':      row['OneBill_AccountType'],
        'activationStartDate': serialize_date(row['CreatedDate']),
        'address': [{
            'addLine1':        row['Address1'],
            'addLine2':        row['Address2'],
            'city':            row['City'],
            'state':           None,
            'country':         'New Zealand',
            'zip':             row['Postcode'],
            'defaultShipping': 'true',
            'defaultBilling':  'true',
        }],
        'contact': [{
            'firstName': row['firstName'],
            'lastName':  row['lastName'],
            'communicationPoint': [
                {'type': 'Email', 'value': row['EmailAddresses']},
                {'type': 'Phone', 'value': row['phoneHome']},
            ],
        }],
        'accountAttribute': [{
            'key':   'Date Of Birth',
            'value': serialize_date(row['DateOfBirth'], fmt='%d/%m/%Y'),
        }],
    })


## Create Account Request

**Bug #2 (Critical):** The original `create_onebill_account` called `get_AccessToken()`
on every request, and also set `Authorization` inside the per-request headers dict —
which overrides the session-level header. We now pull the token from `TokenManager`
and let the session header do the work (updated only on refresh).

In [87]:
def create_onebill_account(session: requests.Session, base_url: str, payload: str) -> dict:
    """POST a single account to OneBill. Reuses session; token comes from TokenManager."""
    url = f'{base_url}/rest/SubscriberService/v1/subscriber'

    # Inject a fresh (possibly cached) token per-request so rotation works mid-batch.
    headers = {'Authorization': f'Bearer {token_manager.get_token()}'}

    response = session.post(url, headers=headers, data=payload, timeout=30)
    response.raise_for_status()
    data = response.json()

    validation = data.get('validationResponse', {})
    if not validation.get('successful', True):
        errors   = validation.get('validationErrorInfo', [])
        messages = '; '.join(e.get('message', '') for e in errors)
        raise ValueError(messages)

    return data


## Worker Function (with per-request timing)

In [88]:
def migrate_row(row: pd.Series, session: requests.Session) -> dict:
    account_code = row['AccountCode']
    first_name = row['firstName']
    last_name = row['lastName']

    # --- Profiling: split timing into build vs. network ---
    t0 = time.perf_counter()
    payload  = build_account_payload(row)
    t_build  = time.perf_counter() - t0

    try:
        t1       = time.perf_counter()
        response = create_onebill_account(session, baseUrl, payload)
        t_net    = time.perf_counter() - t1

        onebill_id = response.get('accountId', 'unknown')
        logger.info(f'  [OK] {account_code} — build={t_build*1000:.0f}ms  net={t_net*1000:.0f}ms')
        return {
            'account_code': account_code,
            'first_name': first_name,
            'last_name': last_name,
            'status':       'success',
            'error':        None,
            'elapsed_build_ms': round(t_build * 1000, 1),
            'elapsed_net_ms':   round(t_net   * 1000, 1),
        }

    except Exception as e:
        t_net = time.perf_counter() - t1 if 't1' in dir() else 0
        logger.error(f'  [FAIL] {account_code} — {e}')
        return {
            'account_code': account_code,
            'first_name': first_name,
            'last_name': last_name,
            'status':       'failed',
            'error':        str(e),
            'elapsed_build_ms': round(t_build * 1000, 1),
            'elapsed_net_ms':   round(t_net   * 1000, 1),
        }


## Migrate Function

**Bug #3 (NameError crash):** In the original code the `futures` dict was built
*before* `session`, `rows`, and `total` were defined. Python executes the dict
comprehension immediately, so it raises `NameError: name 'rows' is not defined`
before any work happens. The variables must be defined first.

**Bug #4 (signature mismatch):** `migrate_row(row, session)` takes 2 arguments but
the original `migrate` passed 3 — `(row, session, token_holder)`. This would raise
`TypeError` at runtime on every submitted future.

**Bug #5 (duplicate `futures` assignment):** The original built `futures` twice —
once before the session existed (crash) and once inside the `with` block. Only the
second one was intended; the first is dead/broken code.

**Bug #6 (timeout too tight):** The original timeout was 10 seconds. If OneBill
takes >10 s under load (not unusual for billing APIs) every request times out.
Increased to 30 s.

**Performance note:** Python's GIL means CPU-bound work doesn't parallelise with
`ThreadPoolExecutor`. For this workload the bottleneck is *network latency*, so
threads help — but only up to the point where the server is the constraint. Check
the profiling output to see whether `elapsed_net_ms` dominates; if so, the server
is rate-limiting or saturated and more workers won't help.

In [89]:
def migrate(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    """Migrate every row in df to OneBill, returning a results DataFrame."""

    # --- FIX #3/#5: define all variables BEFORE building the futures dict ---
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=max_workers,
        pool_maxsize=max_workers
    )
    session.mount('https://', adapter)
    session.headers.update({
        'proxy_accountNumber': '31802',   # Replace with your partner account number
        'Content-Type': 'application/json',
    })

    rows  = [row for _, row in df.iterrows()]
    total = len(rows)
    results: list[dict] = []

    logger.info(f'Starting migration of {total:,} records with {max_workers} workers...')
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # --- FIX #4: only 2 args — token now comes from TokenManager inside migrate_row ---
        futures = {
            executor.submit(migrate_row, row, session): row['AccountCode']
            for row in rows
        }

        for i, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            results.append(result)

            if i % 50 == 0 or i == total:
                ok   = sum(1 for r in results if r['status'] == 'success')
                fail = sum(1 for r in results if r['status'] == 'failed')
                logger.info(f'Progress: {i}/{total} — {ok} ok, {fail} failed')

    wall_elapsed = time.perf_counter() - wall_start

    results_df = pd.DataFrame(results)
    success = (results_df['status'] == 'success').sum()
    failed  = (results_df['status'] == 'failed').sum()

    logger.info(
        f'Migration done in {wall_elapsed:.1f}s — '
        f'{success} succeeded, {failed} failed. (log: {log_filename})'
    )

    # --- Profiling summary ---
    print('\n=== Profiling Summary ===')
    print(f'Total wall time:          {wall_elapsed:.1f}s')
    print(f'Throughput:               {total / wall_elapsed:.1f} accounts/s')
    print(f'Avg build time per row:   {results_df["elapsed_build_ms"].mean():.1f}ms')
    print(f'Avg network time per row: {results_df["elapsed_net_ms"].mean():.1f}ms')
    print(f'Max network time:         {results_df["elapsed_net_ms"].max():.1f}ms')
    print(f'P95 network time:         {results_df["elapsed_net_ms"].quantile(0.95):.1f}ms')
    print('=========================')

    return results_df


## Run Migration

In [90]:
results_df = migrate(df)

failures = results_df[results_df['status'] == 'failed']
print(f'\nFailed rows ({len(failures)}):')
display(failures)


2026-05-05 14:49:05,926 [INFO] Starting migration of 5 records with 20 workers...
2026-05-05 14:49:10,988 [ERROR]   [FAIL] 99999983 — Account number 99999983 already exists.
2026-05-05 14:49:11,069 [ERROR]   [FAIL] 99999994 — Account number 99999994 already exists.; First name is mandatory; Last name is mandatory.; First name is mandatory; Last name is mandatory.
2026-05-05 14:49:11,153 [ERROR]   [FAIL] 99999982 — Account number 99999982 already exists.; First name is mandatory; Last name is mandatory.; First name is mandatory; Last name is mandatory.
2026-05-05 14:49:11,506 [ERROR]   [FAIL] 99999996 — Accounting Name [Accounts Payable] already exist.
2026-05-05 14:49:11,569 [ERROR]   [FAIL] 99999987 — Accounting Name [Accounts Payable] already exist.
2026-05-05 14:49:11,572 [INFO] Progress: 5/5 — 0 ok, 5 failed
2026-05-05 14:49:11,575 [INFO] Migration done in 5.6s — 0 succeeded, 5 failed. (log: migration_20260505_144804.log)



=== Profiling Summary ===
Total wall time:          5.6s
Throughput:               0.9 accounts/s
Avg build time per row:   1.1ms
Avg network time per row: 5319.8ms
Max network time:         5631.9ms
P95 network time:         5620.2ms

Failed rows (5):


,account_code,first_name,last_name,status,error,elapsed_build_ms,elapsed_net_ms
0,99999983,Richard,Conway,failed,Account number 99999983 already exists.,0.7,5048.4
1,99999994,None,None,failed,Account number 99999994 already exists.; First...,0.7,5134.1
2,99999982,None,None,failed,Account number 99999982 already exists.; First...,0.5,5211.1
3,99999996,Accounts,Payable,failed,Accounting Name [Accounts Payable] already exist.,3.2,5573.5
4,99999987,Accounts,Payable,failed,Accounting Name [Accounts Payable] already exist.,0.5,5631.9


In [ ]:
failures.to_csv('Failed_Migrations.csv', index=False)
